# **Netflix Churn Analysis**

**Team:** Netflix Watchers  
**Authors:** Mahek Kothari (@mahekkothari), Kirsten Perez (@kirstenperez)  
**Created:** October 21, 2026  
**Last Modified:** November 14, 2026  

## Problem Statement
Which subscribers are most likely to cancel their Netflix subscription and what behavioral or account-level factors best explain why? Can these risk factors be made into retention strategies?  

## Dataset
- **Source:** [Netflix Customer Churn & Engagement Analytics](https://www.kaggle.com/datasets/zeyadmohamed26/netflix-customer-churn-and-engagement-analytics) (Kaggle, CC0 license)  
- **Size:** 5,000 rows x 14 columns

## Notebook Table of Contents:

**0. Notebook Setup**
- 0.1 Load Libraries
- 0.2 Load the Dataset

**1. Inspecting the Data**
- 1.1 Data Overview
- 1.2 Descriptive Statistics

## **0. Notebook Setup**

### 0.1 Load Libraries

In [1]:
# Import libraries
import pandas as pd
import os
import kagglehub

### 0.2 Load the Dataset

In [2]:
# Download the latest version of the dataset from Kaggle
path = kagglehub.dataset_download("zeyadmohamed26/netflix-customer-churn-and-engagement-analytics")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Kirsten\.cache\kagglehub\datasets\zeyadmohamed26\netflix-customer-churn-and-engagement-analytics\versions\1


In [3]:
# Load the CSV into a DataFrame
df = pd.read_csv(os.path.join(path, "netflix_customer_churn.csv"))

# Show top 5 rows
df.head()

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89,13,Africa,Mobile,13.99,1,Crypto,2,0.13,Action


## **1. Inspecting the Data**

### 1.1 Data Overview

Before analyzing the dataset, we wanted to make sure everything matched what the Kaggle page described (5,000 rows, 14 columns, and no missing values). We also checked that each column has the right data type, that every `customer_id` is unique (so no subscriber shows up twice), and that there are no missing values or duplicate rows. These checks help us decide whether the data needs any cleaning.

In [4]:
# Print the dataset size and column info
print("=== Dataset Size ===")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\n=== Column Info ===")
df.info()

# Verify customer_id is unique (one row per subscriber)
print("\n=== Unique ID Check ===")
print("Unique IDs:", df["customer_id"].nunique())
print("Total rows:", len(df))
print("customer_id is unique:", df["customer_id"].is_unique)

# Verify prices match with plans
print("\n=== Plan Check ===")
display(pd.crosstab(df["subscription_type"], df["monthly_fee"]))

=== Dataset Size ===
Rows: 5000
Columns: 14

=== Column Info ===
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             5000 non-null   str    
 1   age                     5000 non-null   int64  
 2   gender                  5000 non-null   str    
 3   subscription_type       5000 non-null   str    
 4   watch_hours             5000 non-null   float64
 5   last_login_days         5000 non-null   int64  
 6   region                  5000 non-null   str    
 7   device                  5000 non-null   str    
 8   monthly_fee             5000 non-null   float64
 9   churned                 5000 non-null   int64  
 10  payment_method          5000 non-null   str    
 11  number_of_profiles      5000 non-null   int64  
 12  avg_watch_time_per_day  5000 non-null   float64
 13  favorite_genre          5000 non-null  

monthly_fee,8.99,13.99,17.99
subscription_type,,,
Basic,1661,0,0
Premium,0,0,1693
Standard,0,1646,0


In [5]:
# Count missing values for each column
print("=== Missing Values ===")
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())

# Count duplicate rows (dropping customer_id since it's unique)
print("\n=== Duplicate Rows ===")
print("Duplicate rows:", df.drop(columns="customer_id").duplicated().sum())

=== Missing Values ===
customer_id               0
age                       0
gender                    0
subscription_type         0
watch_hours               0
last_login_days           0
region                    0
device                    0
monthly_fee               0
churned                   0
payment_method            0
number_of_profiles        0
avg_watch_time_per_day    0
favorite_genre            0
dtype: int64

Total missing values: 0

=== Duplicate Rows ===
Duplicate rows: 0


**Observations:**

- The dataset matches the Kaggle description: 5,000 rows, 14 columns, and no missing values
- `customer_id` is unique and there are no duplicate rows, so no subscriber appears twice
- Each plan has exactly one price, so `monthly_fee` gives the same information as `subscription_type`:
  - Basic → 8.99 dollars
  - Standard → 13.99 dollars
  - Premium → 17.99 dollars
- All data types are correct, so no conversion is needed

### 1.2 Descriptive Statistics

In [6]:
# Show the counts for each categorical column
for col in ["gender", "subscription_type", "region", "device", "payment_method", "favorite_genre"]:
    print("---", col, "---")
    print(df[col].value_counts().to_string())
    print("Average per category:", round(df[col].value_counts().mean()))
    print()
 
# Show the summary statistics for the numeric columns
df.describe()

--- gender ---
gender
Female    1711
Male      1654
Other     1635
Average per category: 1667

--- subscription_type ---
subscription_type
Premium     1693
Basic       1661
Standard    1646
Average per category: 1667

--- region ---
region
South America    873
Europe           867
North America    851
Asia             841
Africa           803
Oceania          765
Average per category: 833

--- device ---
device
Tablet     1048
Laptop     1006
Mobile     1004
TV          993
Desktop     949
Average per category: 1000

--- payment_method ---
payment_method
Debit Card     1030
PayPal         1026
Crypto          995
Gift Card       976
Credit Card     973
Average per category: 1000

--- favorite_genre ---
favorite_genre
Drama          731
Documentary    729
Romance        725
Sci-Fi         720
Horror         713
Action         697
Comedy         685
Average per category: 714



,age,watch_hours,last_login_days,monthly_fee,churned,number_of_profiles,avg_watch_time_per_day
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,43.847400,11.649450,30.089800,13.683400,0.503000,3.024400,0.874800
std,15.501128,12.014654,17.536078,3.692062,0.500041,1.415841,2.619824
min,18.000000,0.010000,0.000000,8.990000,0.000000,1.000000,0.000000
25%,30.000000,3.337500,15.000000,8.990000,0.000000,2.000000,0.110000
50%,44.000000,8.000000,30.000000,13.990000,1.000000,3.000000,0.290000
75%,58.000000,16.030000,45.000000,17.990000,1.000000,4.000000,0.720000
max,70.000000,110.400000,60.000000,17.990000,1.000000,5.000000,98.420000


**Observations:** Every category has almost the same number of subscribers, which isn't realistic, but it makes sense since the dataset is synthetic. This is a limitation of our analysis since real Netflix data might look different

**Categorical columns**
- `gender`: There are 3 options with an average of 1,667 subscribers each. Female has the most with 1,711 and Other has the least with 1,635
- `subscription_type`: There are 3 plans with an average of 1,667 subscribers each. Premium has the most with 1,693 and Standard has the least with 1,646
- `region`: There are 6 regions with an average of 833 subscribers each. South America has the most with 873 and Oceania has the least with 765
- `device`: There are 5 devices with an average of 1,000 subscribers each. Tablet has the most with 1,048 and Desktop has the least with 949
- `payment_method`: There are 5 payment methods with an average of 1,000 subscribers each. Debit Card has the most with 1,030 and Credit Card has the least with 973
- `favorite_genre`: There are 7 genres with an average of 714 subscribers each. Drama has the most with 731 and Comedy has the least with 685

**Numeric columns**
- `age`: Ages range from 18-70 and are spread out evenly with the mean and median about the same (around 44)
- `watch_hours`: The mean (11.65) is higher than the median (8.0) because a small group of heavy watchers (up to 110.4 hours) pulls the average up. Most subscribers watch much less than the mean suggests
- `last_login_days`: Days since last login range from 0 to 60 and are spread out evenly with the mean and median about the same (around 30)
- `monthly_fee`: There are only 3 options (8.99 dollars, 13.99 dollars, and 17.99 dollars) with 1 for each plan, so this column gives the same information as `subscription_type`
- `churned`: There are only 2 options (0 = stayed and 1 = cancelled). The mean is 0.50, so about half of subscribers cancelled and there are enough of both groups to compare fairly
- `number_of_profiles`: There are only 5 options (1, 2, 3, 4, and 5) since an account can have up to 5 profiles
- `avg_watch_time_per_day`: The max is 98.42 hours a day, which is impossible since a day only has 24 hours. The Kaggle page says the range should be 0.01 to 4.97, so this column will be removed during cleaning